In [9]:
# ==============================================================================
# Cell 1: Setup, Paths, and Machine Learning Dependencies (GCP Optimized)
# ==============================================================================

"""
Configuración del Espacio Léxico-Semántico Reducido (Latent Semantic Analysis).
Importa dependencias y prepara el modelo Transformer (TRF) de spaCy.

OPTIMIZACIÓN GCP/TRF:
Se deshabilitan los componentes 'parser' y 'ner' para maximizar la velocidad 
de inferencia. Si se ejecuta en una instancia de Google Cloud con GPU, spaCy 
automáticamente intentará utilizarla para acelerar el procesamiento con PyTorch.
"""

import os
import re
import pickle
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import tgt
import spacy
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

# 1. Configuración de Aceleración por Hardware (GPU si está disponible en GCP)
spacy.prefer_gpu()

# 2. Rutas y Checkpoints
BASE_DIR = Path("/home/amont21/Documentos/voxelwise modeling/ds003020/derivatives/TextGrids")
CACHE_DATA_PATH = "lemmatized_corpus_cache.pkl"
CACHE_MODEL_PATH = "semantic_lsa_model.joblib"

# 3. Carga de Modelo NLP
SPACY_MODEL_NAME = "en_core_web_trf"
print(f"Cargando modelo de spaCy: {SPACY_MODEL_NAME}...")

# Apagamos el parser y el ner (No se necesitan para LSA, ahorra 50% de RAM/CPU)
nlp = spacy.load(SPACY_MODEL_NAME, disable=["parser", "ner"]) 

# ------------------------------------------------------------------------------
# INYECCIÓN DE STOPWORDS CONVERSACIONALES
# ------------------------------------------------------------------------------
# Ampliamos el diccionario nativo de spaCy con light verbs del habla espontánea
# descubiertos durante el Análisis Exploratorio de Datos (EDA).
SPOKEN_STOP_WORDS = {
    'like', 'know', 'go', 'uh', 'say', 'um', 'think', 'get', 
    'come', 'look', 'thing', 'tell', 'want', 'start', 'feel', 
    'right', 'mean', 'kind', 'yeah', 'oh', 'well'
}

for word in SPOKEN_STOP_WORDS:
    nlp.vocab[word].is_stop = True

print(f"✅ Se añadieron {len(SPOKEN_STOP_WORDS)} stopwords conversacionales (light verbs) a spaCy.")

# 4. Parámetros del Espacio Semántico
HIGH_RES_FS = 100 
THRESHOLD_PERIOD_SEC = 0.65  
N_LATENT_COMPONENTS = 10

Cargando modelo de spaCy: en_core_web_trf...
✅ Se añadieron 21 stopwords conversacionales (light verbs) a spaCy.


In [10]:
# ==============================================================================
# Cell 2: Global Semantic Trainer and Topic Extraction (CPU/GPU Safe)
# ==============================================================================

"""
Entrena el modelo global de Análisis Semántico Latente (LSA) sobre el corpus.
Implementa un flujo de carga/guardado para favorecer la iteración metodológica.
NOTA TÉCNICA: Se ejecuta en un solo hilo (sin multiprocessing) para evitar 
deadlocks en Jupyter, dado que el tamaño del corpus (aprox. 7000 oraciones) 
se procesa en segundos de forma secuencial.

OPTIMIZACIÓN TRF: 
Se utiliza n_process=1 en el pipeline de spaCy. Esto es obligatorio para 
modelos basados en Transformers (PyTorch) cuando se ejecutan en CPU, ya que 
PyTorch maneja su propio multithreading (OpenMP/MKL) a nivel de operaciones 
tensoriales. Evita bloqueos (deadlocks) del sistema.
"""

def extract_sentences_from_corpus(base_dir: Path) -> List[str]:
    """Extrae oraciones de las historias basándose en pausas largas."""
    textgrid_files = [f for f in base_dir.rglob("*.TextGrid") if f.name not in ['legacy.TextGrid', 'exorcism.TextGrid']]
    all_sentences = []
    noise_pattern = re.compile(r'\[|\]|\{|\}|\<|\>|spn|^sp$|^sil$|^br$|^lg$', re.IGNORECASE)
    
    for file_path in textgrid_files:
        try:
            tg = tgt.io.read_textgrid(str(file_path), include_empty_intervals=True)
            word_tier = next((t for t in tg.get_tier_names() if 'word' in t.lower()), None)
            if not word_tier: continue
                
            current_sentence = []
            for interval in tg.get_tier_by_name(word_tier).intervals:
                token = interval.text.strip()
                if token == "" or bool(noise_pattern.search(token)):
                    duration = interval.end_time - interval.start_time
                    if duration >= THRESHOLD_PERIOD_SEC and current_sentence:
                        all_sentences.append(" ".join(current_sentence))
                        current_sentence = []
                    continue
                
                clean_word = re.sub(r'[^a-zA-Z\']', '', token).lower()
                if clean_word:
                    current_sentence.append(clean_word)
                    
            if current_sentence:
                all_sentences.append(" ".join(current_sentence))
        except Exception:
            pass
            
    return all_sentences

def preprocess_for_semantics(doc: spacy.tokens.Doc) -> str:
    """Extrae lemas y remueve stop-words de un documento ya procesado."""
    lemmas = [
        token.lemma_.lower() for token in doc 
        if not token.is_stop and not token.is_punct and not token.like_num and token.lemma_.strip()
    ]
    return " ".join(lemmas)

# ==============================================================================
# FLUJO PRINCIPAL CON PUNTOS DE GUARDADO (CHECKPOINTS)
# ==============================================================================
if os.path.exists(CACHE_DATA_PATH) and os.path.exists(CACHE_MODEL_PATH):
    print("🔄 ¡Caché detectada! Cargando datos y modelo guardados...")
    with open(CACHE_DATA_PATH, 'rb') as f:
        valid_sentences = pickle.load(f)
    semantic_pipeline = joblib.load(CACHE_MODEL_PATH)
    print("✅ Carga completada exitosamente.")

else:
    print("⚠️ No se encontró caché. Iniciando procesamiento desde cero...")
    
    print("1. Extrayendo oraciones del corpus...")
    raw_sentences = extract_sentences_from_corpus(BASE_DIR)
    print(f"   -> Se encontraron {len(raw_sentences)} oraciones temporales.")

    print("\n2. Lematizando con Transformer (Optimizando uso de tensores)...")
    
    # n_process=1 evita deadlocks en CPU. batch_size=256 optimiza RAM.
    processed_sentences = []
    
    # Procesamos e imprimimos el progreso para que sepas que no está bloqueado
    for i, doc in enumerate(nlp.pipe(raw_sentences, batch_size=256, n_process=1)):
        processed_sentences.append(preprocess_for_semantics(doc))
        if (i + 1) % 1000 == 0:
            print(f"   ... Procesadas {i + 1} de {len(raw_sentences)} oraciones")
            
    valid_sentences = [s for s in processed_sentences if s.strip()]
    
    print("\n   💾 Guardando lemas limpios en caché...")
    with open(CACHE_DATA_PATH, 'wb') as f:
        pickle.dump(valid_sentences, f)

    print("\n3. Entrenando Modelo Global LSA (TF-IDF + SVD)...")
    semantic_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(min_df=3, max_df=0.85)),
        ('svd', TruncatedSVD(n_components=N_LATENT_COMPONENTS, random_state=42))
    ])
    
    semantic_pipeline.fit(valid_sentences)
    
    print("   💾 Guardando modelo de Machine Learning en caché...")
    joblib.dump(semantic_pipeline, CACHE_MODEL_PATH)
    
    print("✅ Entrenamiento y guardado completados.")

# ==============================================================================
# Interpretación de Componentes Latentes
# ==============================================================================
print("\n" + "="*60)
print("🔍 INTERPRETACIÓN DE LOS COMPONENTES SEMÁNTICOS (TÓPICOS REALES)")
print("="*60)

tfidf_model = semantic_pipeline.named_steps['tfidf']
svd_model = semantic_pipeline.named_steps['svd']
feature_names = tfidf_model.get_feature_names_out()

for i, component in enumerate(svd_model.components_):
    top_indices = component.argsort()[::-1][:8]
    top_words = [feature_names[idx] for idx in top_indices]
    
    print(f"▶ Componente {i}:")
    print(f"  {', '.join(top_words)}")

🔄 ¡Caché detectada! Cargando datos y modelo guardados...
✅ Carga completada exitosamente.

🔍 INTERPRETACIÓN DE LOS COMPONENTES SEMÁNTICOS (TÓPICOS REALES)
▶ Componente 0:
  time, day, year, people, old, good, life, little
▶ Componente 1:
  day, school, thank, later, good, new, remember, week
▶ Componente 2:
  time, day, remember, spend, early, particular, long, plan
▶ Componente 3:
  thank, time, god, sit, alright, moment, child, fine
▶ Componente 4:
  year, old, time, day, later, thank, ago, school
▶ Componente 5:
  ask, happen, day, year, question, stand, time, dad
▶ Componente 6:
  happen, people, love, life, find, talk, thank, moment
▶ Componente 7:
  love, way, ask, hand, good, find, write, little
▶ Componente 8:
  people, love, little, mother, day, ask, talk, work
▶ Componente 9:
  way, people, talk, week, work, new, life, ask


In [11]:
# ==============================================================================
# Cell 2.5: Semantic Cache Audit and Custom Stop-Words Discovery
# ==============================================================================

"""
Audita el corpus lematizado previamente guardado para identificar muletillas 
(fillers) y palabras hiperfrecuentes que spaCy no eliminó por defecto.
Esto es vital para garantizar que el Análisis Semántico Latente (LSA) capture 
'contenido temático' y no simples artefactos del habla espontánea.
"""

import pickle
import pandas as pd
from collections import Counter

CACHE_DATA_PATH = "lemmatized_corpus_cache.pkl"

def audit_lemmatized_cache():
    try:
        with open(CACHE_DATA_PATH, 'rb') as f:
            valid_sentences = pickle.load(f)
            
        print(f"✅ Caché cargada: {len(valid_sentences)} oraciones.")
        
        # Contar todas las palabras en el corpus lematizado
        all_lemmas = []
        for sentence in valid_sentences:
            all_lemmas.extend(sentence.split())
            
        lemma_counts = Counter(all_lemmas)
        
        # Crear DataFrame para el EDA
        df_lemmas = pd.DataFrame.from_dict(lemma_counts, orient='index', columns=['Frequency'])
        df_lemmas.index.name = 'Lemma'
        df_lemmas = df_lemmas.sort_values(by='Frequency', ascending=False)
        
        print("\n" + "="*50)
        print("🔍 TOP 40 LEMAS MÁS FRECUENTES EN EL CORPUS")
        print("="*50)
        print("Revisa esta lista para identificar muletillas (um, uh, like) o ")
        print("palabras vacías (to, so) que debemos filtrar manualmente.")
        
        display(df_lemmas.head(40))
        
        return valid_sentences
        
    except FileNotFoundError:
        print("❌ Error: No se encontró el archivo de caché. Debes correr la Celda 2 primero.")
        return None

# Ejecutar auditoría
cached_sentences = audit_lemmatized_cache()

✅ Caché cargada: 6685 oraciones.

🔍 TOP 40 LEMAS MÁS FRECUENTES EN EL CORPUS
Revisa esta lista para identificar muletillas (um, uh, like) o 
palabras vacías (to, so) que debemos filtrar manualmente.


,Frequency
Lemma,
go,997
say,752
get,489
to,419
time,413
year,347
day,336
think,311
look,301


In [12]:
# ==============================================================================
# Cell 2.6: Custom Stop-Words Filtering and LSA Re-training
# ==============================================================================

"""
Filtra empíricamente las muletillas y verbos ligeros descubiertos en la auditoría,
y re-entrena el modelo de Análisis Semántico Latente (LSA) para que capture 
verdaderos 'tópicos' abstractos y no artefactos del habla espontánea.
"""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
import joblib

# 1. Definir lista negra basada en la auditoría empírica
CUSTOM_STOP_WORDS = {
    'like', 'know', 'go', 'uh', 'say', 'um', 'think', 'get', 
    'come', 'look', 'to', 'thing', 'tell', 'want', 'start', 
    'feel', 'right', 'mean', 'kind', 'yeah', 'oh', 'well'
}

print("1. Aplicando filtro de palabras vacías personalizadas...")
clean_sentences = []
for sentence in cached_sentences: # cached_sentences viene de la Celda 2.5
    # Mantener solo las palabras que NO están en la lista negra
    filtered_words = [word for word in sentence.split() if word not in CUSTOM_STOP_WORDS]
    if filtered_words:
        clean_sentences.append(" ".join(filtered_words))

print(f"   -> Oraciones válidas tras la limpieza: {len(clean_sentences)}")

print("\n2. Re-entrenando Modelo Global LSA (TF-IDF + SVD)...")
semantic_pipeline = Pipeline([
    # min_df=3 significa que la palabra debe aparecer en al menos 3 oraciones distintas
    ('tfidf', TfidfVectorizer(min_df=3, max_df=0.85)),
    ('svd', TruncatedSVD(n_components=10, random_state=42))
])

semantic_pipeline.fit(clean_sentences)

# Sobreescribimos el modelo anterior en el disco duro
joblib.dump(semantic_pipeline, "semantic_lsa_model.joblib")
print("   ✅ Nuevo modelo guardado en caché.")

# ==============================================================================
# Interpretación de Componentes Latentes (Versión Limpia)
# ==============================================================================
print("\n" + "="*60)
print("🔍 NUEVOS COMPONENTES SEMÁNTICOS (TÓPICOS REALES)")
print("="*60)

tfidf_model = semantic_pipeline.named_steps['tfidf']
svd_model = semantic_pipeline.named_steps['svd']
feature_names = tfidf_model.get_feature_names_out()

for i, component in enumerate(svd_model.components_):
    top_indices = component.argsort()[::-1][:8]
    top_words = [feature_names[idx] for idx in top_indices]
    
    print(f"▶ Componente {i}:")
    print(f"  {', '.join(top_words)}")

1. Aplicando filtro de palabras vacías personalizadas...
   -> Oraciones válidas tras la limpieza: 6579

2. Re-entrenando Modelo Global LSA (TF-IDF + SVD)...
   ✅ Nuevo modelo guardado en caché.

🔍 NUEVOS COMPONENTES SEMÁNTICOS (TÓPICOS REALES)
▶ Componente 0:
  time, day, year, people, old, good, life, little
▶ Componente 1:
  day, school, thank, later, good, new, remember, week
▶ Componente 2:
  time, day, remember, spend, early, particular, long, plan
▶ Componente 3:
  thank, time, god, sit, alright, moment, child, fine
▶ Componente 4:
  year, old, time, day, later, thank, ago, school
▶ Componente 5:
  ask, happen, day, year, question, stand, time, dad
▶ Componente 6:
  happen, people, love, life, find, talk, thank, moment
▶ Componente 7:
  love, way, ask, hand, good, find, write, little
▶ Componente 8:
  people, love, little, mother, day, ask, talk, work
▶ Componente 9:
  way, people, talk, week, work, new, life, ask


In [13]:
# ==============================================================================
# Cell 3: Temporal Projection of Semantic Latent Space
# ==============================================================================

"""
Genera la matriz temporal (100 Hz) del Espacio Léxico-Semántico Reducido.

JUSTIFICACIÓN METODOLÓGICA (Dinámica Semántica Temporal):
El significado en el habla naturalista se construye proposición a proposición.
Por ello, el espacio temporal se construye segmentando el flujo de habla en 
cláusulas (utilizando el umbral empírico de pausas >= 0.65s). 
Cada cláusula se proyecta en el espacio LSA de 10 dimensiones pre-entrenado, 
y estos valores (pesos semánticos) se sostienen en la matriz temporal desde 
el inicio de la primera palabra de la cláusula hasta el final de la última, 
reflejando el estado semántico mantenido en la memoria de trabajo del oyente.

JUSTIFICACIÓN METODOLÓGICA (Dinámica Semántica Temporal):
El significado se construye proposición a proposición. El espacio temporal 
se construye segmentando el flujo de habla en cláusulas (pausas >= 0.65s). 
Cada cláusula se proyecta en el espacio LSA de 10 dimensiones pre-entrenado.
Estos pesos semánticos se sostienen en la matriz temporal desde el inicio 
de la primera palabra hasta el final de la última, reflejando el estado 
semántico mantenido en la memoria de trabajo del oyente.
"""

from typing import Union
import numpy as np
import pandas as pd
import tgt
from pathlib import Path
import re
import spacy
from sklearn.pipeline import Pipeline

# Nombres de las columnas de tu matriz final
SEMANTIC_FEATURE_NAMES = [f"semantic_dim_{i}" for i in range(N_LATENT_COMPONENTS)]

# Definimos de nuevo CUSTOM_STOP_WORDS para que esta función pueda usarlas si lematiza sobre la marcha
CUSTOM_STOP_WORDS = {
    'like', 'know', 'go', 'uh', 'say', 'um', 'think', 'get', 
    'come', 'look', 'to', 'thing', 'tell', 'want', 'start', 
    'feel', 'right', 'mean', 'kind', 'yeah', 'oh', 'well'
}

def extract_semantic_space(textgrid_path: Union[str, Path], semantic_model: Pipeline, nlp_model: spacy.language.Language, fs: int) -> pd.DataFrame:
    """
    Proyecta los pesos del LSA en el tiempo continuo de la historia.
    """
    tg = tgt.io.read_textgrid(str(textgrid_path), include_empty_intervals=True)
    word_tier = next((t for t in tg.get_tier_names() if 'word' in t.lower()), None)
    
    if not word_tier:
        raise ValueError(f"No se encontró capa 'words' en {textgrid_path}")
        
    total_duration = tg.get_tier_by_name(word_tier).end_time
    total_samples = int(np.ceil(total_duration * fs))
    
    # float32 porque los pesos SVD son continuos
    feature_matrix = np.zeros((total_samples, len(SEMANTIC_FEATURE_NAMES)), dtype=np.float32)
    
    noise_pattern = re.compile(r'\[|\]|\{|\}|\<|\>|spn|^sp$|^sil$|^br$|^lg$', re.IGNORECASE)
    
    current_sentence_words = []
    sentence_start_time = None
    sentence_end_time = None
    
    intervals = tg.get_tier_by_name(word_tier).intervals
    
    for i, interval in enumerate(intervals):
        token = interval.text.strip()
        is_empty = token == ""
        is_noise = bool(noise_pattern.search(token))
        
        # 1. Evaluación de pausas (Fin de oración)
        if is_empty or is_noise:
            duration = interval.end_time - interval.start_time
            if duration >= THRESHOLD_PERIOD_SEC and current_sentence_words:
                
                raw_sentence = " ".join(current_sentence_words)
                
                # Preprocesamiento ajustado: procesamos y limpiamos las stopwords personalizadas
                doc = nlp_model(raw_sentence)
                lemmas = [
                    t.lemma_.lower() for t in doc 
                    if not t.is_stop and not t.is_punct and not t.like_num and t.lemma_.strip()
                ]
                final_lemmas = [w for w in lemmas if w not in CUSTOM_STOP_WORDS]
                lemmatized_sentence = " ".join(final_lemmas)
                
                if lemmatized_sentence.strip():
                    # Transformar al espacio de 10 dimensiones
                    vector_10d = semantic_model.transform([lemmatized_sentence])[0]
                    
                    # Proyectar en la matriz temporal
                    start_idx = int(np.floor(sentence_start_time * fs))
                    end_idx = int(np.ceil(sentence_end_time * fs))
                    
                    feature_matrix[start_idx:end_idx, :] = vector_10d
                    
                current_sentence_words = []
                sentence_start_time = None
            continue
            
        # 2. Recopilación de palabras
        clean_word = re.sub(r'[^a-zA-Z\']', '', token).lower()
        if clean_word:
            if not current_sentence_words:
                sentence_start_time = interval.start_time
            sentence_end_time = interval.end_time
            current_sentence_words.append(clean_word)
            
    # Capturar la última oración
    if current_sentence_words and sentence_start_time is not None:
        raw_sentence = " ".join(current_sentence_words)
        doc = nlp_model(raw_sentence)
        lemmas = [
            t.lemma_.lower() for t in doc 
            if not t.is_stop and not t.is_punct and not t.like_num and t.lemma_.strip()
        ]
        final_lemmas = [w for w in lemmas if w not in CUSTOM_STOP_WORDS]
        lemmatized_sentence = " ".join(final_lemmas)
        
        if lemmatized_sentence.strip():
            vector_10d = semantic_model.transform([lemmatized_sentence])[0]
            start_idx = int(np.floor(sentence_start_time * fs))
            end_idx = int(np.ceil(sentence_end_time * fs))
            end_idx = min(end_idx, total_samples)
            feature_matrix[start_idx:end_idx, :] = vector_10d
            
    time_axis = np.arange(total_samples) / fs
    df_features = pd.DataFrame(feature_matrix, columns=SEMANTIC_FEATURE_NAMES, index=time_axis)
    df_features.index.name = 'time_seconds'
    
    return df_features

In [14]:
# ==============================================================================
# Cell 4: Execution and Semantic Space Visualization
# ==============================================================================

"""
Prueba el motor semántico extrayendo la matriz temporal de una historia.
Verificaremos cómo los tópicos (dimensiones) fluctúan en el tiempo.
"""

try:
    if 'semantic_pipeline' in locals():
        textgrid_files = list(BASE_DIR.rglob("*.TextGrid"))
        valid_file = next((f for f in textgrid_files if f.name not in ['legacy.TextGrid', 'exorcism.TextGrid']), None)
                
        if valid_file:
            print(f"Extrayendo dinámica semántica (10 Componentes) para: {valid_file.name}")
            
            df_semantic_space = extract_semantic_space(valid_file, semantic_pipeline, nlp, HIGH_RES_FS)
            
            print("\nInformación del Espacio Léxico-Semántico Reducido:")
            print(f"  -> Dimensiones (Muestras x Rasgos): {df_semantic_space.shape}")
            print(f"  -> Tipo de dato: {df_semantic_space.values.dtype}\n")
            
            print("Visualización (Nótese cómo los pesos semánticos cambian bloque a bloque):")
            
            # Filtramos instantes activos
            active_sem_samples = df_semantic_space[(df_semantic_space.sum(axis=1) != 0)]
            
            display(active_sem_samples.drop_duplicates().head(10))
            
        else:
            print("No se encontraron archivos válidos.")
    else:
        print("⏳ El modelo LSA no está cargado.")

except Exception as e:
    print(f"Ocurrió un error en la extracción semántica: {str(e)}")

Extrayendo dinámica semántica (10 Componentes) para: odetostepfather.TextGrid

Información del Espacio Léxico-Semántico Reducido:
  -> Dimensiones (Muestras x Rasgos): (82811, 10)
  -> Tipo de dato: float32

Visualización (Nótese cómo los pesos semánticos cambian bloque a bloque):


,semantic_dim_0,semantic_dim_1,semantic_dim_2,semantic_dim_3,semantic_dim_4,semantic_dim_5,semantic_dim_6,semantic_dim_7,semantic_dim_8,semantic_dim_9
time_seconds,,,,,,,,,,
0.03,0.238912,0.086996,-0.088179,-0.033168,0.019336,-0.043095,0.056659,-0.093408,0.176322,0.114949
34.51,0.180390,0.017327,-0.082866,-0.033430,-0.090162,-0.091684,-0.167490,-0.081534,0.031376,-0.194419
45.70,0.024564,-0.000437,-0.018609,-0.005449,-0.002782,0.002938,-0.008820,-0.018539,0.011380,0.005824
51.08,0.103141,0.001558,-0.053010,-0.011352,-0.044344,-0.022936,-0.039545,-0.020339,0.003689,-0.021287
161.11,0.109477,-0.008940,-0.038057,-0.010874,-0.084622,-0.044769,-0.105803,0.004537,0.093631,-0.076510
166.78,0.087832,0.002355,-0.074314,0.001157,-0.083233,-0.110670,0.126146,0.386364,0.207345,-0.340104
169.73,0.250296,-0.284068,0.187845,0.045353,-0.022072,-0.058842,0.114851,0.326909,0.179042,-0.262662
177.46,0.105860,-0.005616,-0.057682,-0.009794,-0.069161,-0.046277,-0.026214,0.033252,-0.045213,0.052634
199.92,0.085188,0.000593,-0.046756,-0.002520,-0.042157,-0.028380,-0.083477,0.010102,0.000017,-0.008806


In [ ]:
"""
¡No veo absolutamente nada extraño! De hecho, veo la confirmación visual de que la matemática está funcionando a la perfección.
Te explico lo que estás viendo: los saltos de tiempo (0.03, 34.51, 45.70, 51.08) son exactamente los instantes en los que el narrador terminó una oración (hizo una pausa > 0.65s) y comenzó una nueva. Durante esos bloques de tiempo, el cerebro del oyente mantiene ese "estado semántico" (los valores de los 10 componentes) en su memoria de trabajo, y luego, en la siguiente oración, el estado cambia. Los valores entre -0.3 y +0.5 son el estándar matemático de la proyección SVD. ¡Es una matriz hermosa y lista para el modelo predictivo!
"""